In [1]:
import functools
import numpy as np
import os
import random
import matplotlib.pyplot as plt
from __future__ import annotations
import glob
import time
import supersuit as ss
import cv2

import gymnasium
from gymnasium.spaces import MultiDiscrete, Discrete, Dict, Box

from pettingzoo import ParallelEnv
from pettingzoo.test import parallel_api_test
from pettingzoo.utils import parallel_to_aec, wrappers

from stable_baselines3 import PPO
from stable_baselines3.ppo import MultiInputPolicy

In [2]:
def create_video_from_pngs(images_folder):

    # Output video file name
    output_video = "output_video.mp4"

    # Get a list of all PNG files in the folder and their creation times
    image_files_with_dates = [(os.path.join(images_folder, file), os.stat(os.path.join(images_folder, file)).st_mtime) for file in os.listdir(images_folder) if file.endswith(".png")]

    # Sort the files based on creation date
    image_files_sorted = sorted(image_files_with_dates, key=lambda x: x[1])

    # Extract file paths from the sorted list
    image_files = [file[0] for file in image_files_sorted]

    # Read the first image to get dimensions
    first_image = cv2.imread(image_files[0])
    height, width, layers = first_image.shape

    # Define the codec and create VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")  # Codec for MP4 format
    fps = 0.8  # Decrease the frame rate to increase the delay between frames
    video = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

    # Iterate through each image and write it to the video
    for image_file in image_files:
        image = cv2.imread(image_file)
        video.write(image)

    # Release the video object
    video.release()

    print(f"Video created successfully: {output_video}")



def train(
    env_fn, steps: int = 10_000, learning_rate=1e-3, seed: int | None = 0, **env_kwargs
):
    # Train a single model to play as each agent in a cooperative Parallel environment
    
    #env = env_fn.parallel_env(**env_kwargs)
    
    env = env_fn(**env_kwargs)

    env.reset(seed=seed)

    print(f"Starting training on {str(env.metadata['name'])}.")

    env = ss.pettingzoo_env_to_vec_env_v1(env)
    env = ss.concat_vec_envs_v1(env, 8, num_cpus=8, base_class="stable_baselines3")

    # Note: Waterworld's observation space is discrete (242,) so we use an MLP policy rather than CNN
    model = PPO(
        MultiInputPolicy,
        env,
        verbose=3,
        learning_rate=learning_rate,
        batch_size=256,
        ent_coef=0.5,  # Adjust the value as needed

    )

    model.learn(total_timesteps=steps)

    model.save(f"{env.unwrapped.metadata.get('name')}_{time.strftime('%Y%m%d-%H%M%S')}")

    print("Model has been saved.")

    print(f"Finished training on {str(env.unwrapped.metadata['name'])}.")

    env.close()


def evaluate(env_fn, num_games: int = 100, render_mode: str | None = None, **env_kwargs):
    
    # Evaluate a trained agent vs a random agent
    env = env_fn(render_mode=render_mode, **env_kwargs)
    env = parallel_to_aec(env)

    print(
        f"\nStarting evaluation on {str(env.metadata['name'])} (num_games={num_games}, render_mode={render_mode})"
    )

    try:
        latest_policy = max(
            glob.glob(f"{env.metadata['name']}*.zip"), key=os.path.getctime
        )
    except ValueError:
        print("Policy not found.")
        exit(0)

    model = PPO.load(latest_policy)

    rewards = {agent: 0 for agent in env.possible_agents}

    # Note: We train using the Parallel API but evaluate using the AEC API
    # SB3 models are designed for single-agent settings, we get around this by using he same model for every agent
    
    for i in range(num_games):
        env.reset(seed=i)

        for agent in env.agent_iter():
            
            obs, reward, termination, truncation, info = env.last()
            if not termination or not truncation:
                if agent == env.agents[-1]: env.render()

            for a in env.agents:
                rewards[a] += env.rewards[a]
            if termination or truncation:
                break
            else:
                act = model.predict(obs, deterministic=True)[0]

            env.step(act)
            
        if not termination or not truncation:env.render()
            
    env.close()

    avg_reward = sum(rewards.values()) / len(rewards.values())
    print("Rewards: ", rewards)
    print(f"Avg reward: {avg_reward}")
    return avg_reward


In [3]:
#Trying to follow https://pettingzoo.farama.org/content/environment_creation/#example-custom-parallel-environment

In [45]:
class ReverseAuctionEnv(ParallelEnv):

    metadata = {"render_modes": ["human"], "name": "reverse_auction_v1"}

    def __init__(self, render_mode="human", num_bidders=5, max_rounds=20):
        self.possible_agents = ["provider_" + str(r) for r in range(num_bidders)]
        self.agent_name_mapping = dict(zip(self.possible_agents, list(range(len(self.possible_agents)))))
        self.render_mode = render_mode
        self.max_rounds = max_rounds
        self.round = 1
        self.done = False
        self.initial_bids = np.random.randint(55, 60, size=num_bidders).astype(np.float64)        
        self.bids = self.initial_bids.copy()
        self.prev_ranks = np.ones(num_bidders, dtype=int)

    @functools.lru_cache(maxsize=None)
    def observation_space(self, agent):
        return Dict({
            'current_rank': Discrete(len(self.possible_agents) + 1),
            'previous_rank': Discrete(len(self.possible_agents) + 1),
            'current_round': Discrete(self.max_rounds + 1)
        })

    @functools.lru_cache(maxsize=None)
    def action_space(self, agent):
        return Discrete(9)  # 0->same bid, 1-4>higher bid, 5-8>lower bid

    
    def render(self):
        if self.render_mode == "human":
            num_agents = len(self.possible_agents)
            round_number = self.round
            plt.figure(figsize=(10, 6))
    
            for i in range(num_agents):
                agent_bid = self.bids[i]
                plt.plot([i + 1], [agent_bid], marker='o', label=f"{self.possible_agents[i]} Bid")
            
            plt.xlabel("Agents")
            plt.ylabel("Bid Value")
            plt.ylim(0, 100)  # Set y-axis boundaries
            plt.title(f"Bids of Each Agent in Round {round_number}")
            plt.legend()
            plt.grid(True)
            
            output_folder = "outputs"
            if not os.path.exists(output_folder):
                os.makedirs(output_folder)
    
            output_path = os.path.join(output_folder, f"round_{round_number}_bids.png")
            plt.savefig(output_path)
            plt.close()
            
        
    def close(self):
        pass

    def reset(self, seed=None, options=None):
        self.agents = self.possible_agents[:]
        self.round = 1
        self.done = False
        self.initial_bids = np.random.randint(55, 60, size=len(self.possible_agents)).astype(np.float64)  
        self.bids = self.initial_bids.copy()
        self.prev_ranks = np.ones(len(self.agents), dtype=int)

        rank = 1

        observations = {agent: {'current_rank': rank, 'previous_rank': rank, 'current_round': self.round} for agent in self.agents}
        self.state = observations
        infos = {agent: {} for agent in self.agents}
        return observations, infos

    def step(self, actions):
        if not actions:
            #self.agents = []
            return {}, {}, {}, {}, {}

        if self.round == self.max_rounds:
            self.done = True

        rewards = {}
        observations = {}
        
        for agent in self.agents:
            action = actions[agent]
            agent_id = self.agent_name_mapping[agent]

            if action == 0:  # Same bid
                pass
            elif 1 <= action <= 4:  # Higher bid
                self.bids[agent_id] *= 1.1 + (action - 1) * 0.1
            elif 5 <= action <= 8:  # Lower bid
                self.bids[agent_id] *= 0.9 - (action - 5) * 0.1


        sorted_indices = np.argsort(self.bids)
        ranks = np.empty_like(sorted_indices)
        ranks[sorted_indices] = np.arange(len(sorted_indices)) + 1

        for agent in self.agents:
            agent_id = self.agent_name_mapping[agent]
            previous_rank = self.prev_ranks[agent_id]
            current_rank = ranks[agent_id]

            rewards[agent] = 0
            
            # Reward for improving rank
            if current_rank < previous_rank:
                rewards[agent] += 1  
            # Negative reward for worsening rank
            elif current_rank > previous_rank:
                rewards[agent] -= 1  

            
            # Negative reward for bid outside range
            if not (30 < self.bids[agent_id] < 60):
                rewards[agent] -= 30  
            else:
                rewards[agent] += 30
            
            # Reward for being first
            if current_rank == 1:
                if (30 < self.bids[agent_id] < 40): rewards[agent] += 3 
                else: rewards[agent] -= 1
            # Negative reward for not being first in the last round
            elif self.round == self.max_rounds and current_rank != 1:
                rewards[agent] -= 1
            
            observations[agent] = {
                'current_rank': current_rank,
                'previous_rank': previous_rank,
                'current_round': self.round
            }
            
            self.prev_ranks[agent_id] = current_rank


        self.state = observations

        infos = {agent: {} for agent in self.agents}
        terminations = {agent: self.done for agent in self.agents}
        truncations = {agent: self.done for agent in self.agents}

        self.round += 1

        if self.done:
            self.agents = [] 

        return observations, rewards, terminations, truncations, infos

In [46]:
if __name__ == "__main__":
    
    env = ReverseAuctionEnv()
    parallel_api_test(env, num_cycles=10_000)

    env_fn = ReverseAuctionEnv
    
    env_kwargs = {}

    # Train a model (takes ~3 minutes on GPU)
    train(env_fn, steps=200_000, learning_rate=1e-3 , seed=42, **env_kwargs)
    

Starting training on reverse_auction_v1.
Using cpu device
------------------------------
| time/              |       |
|    fps             | 20709 |
|    iterations      | 1     |
|    time_elapsed    | 3     |
|    total_timesteps | 81920 |
------------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 9109        |
|    iterations           | 2           |
|    time_elapsed         | 17          |
|    total_timesteps      | 163840      |
| train/                  |             |
|    approx_kl            | 0.012682286 |
|    clip_fraction        | 0.159       |
|    clip_range           | 0.2         |
|    entropy_loss         | -2.19       |
|    explained_variance   | 1.97e-05    |
|    learning_rate        | 0.001       |
|    loss                 | 8.26e+03    |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.0279     |
|    value_loss           | 1.74e+04    |
-----------

In [47]:
env_fn = ReverseAuctionEnv
    
env_kwargs = {}

evaluate(env_fn, num_games=1, render_mode="human", **env_kwargs)

images_folder = "outputs"
create_video_from_pngs(images_folder)



Starting evaluation on reverse_auction_v1 (num_games=1, render_mode=human)
Rewards:  {'provider_0': 362, 'provider_1': 411, 'provider_2': 241, 'provider_3': 230, 'provider_4': 170}
Avg reward: 282.8
Video created successfully: output_video.mp4
